## SQL Pipeline

Pipeline stages:
1. **Check data** - Look at data structure of Autumn 2025 pupils and schools data
2. **Data Manipulation** - Select columns and metadata, join pupils to schools
3. **Outputs** - Produce avg pupil premium by Ofsted rating summary

In [0]:
USE CATALOG catalog_40_copper_analyst_training;

### Check data
Look at data structure.

In [0]:
-- pupils autumn 2025
SELECT * FROM catalog_40_copper_analyst_training.bronze.pupils_autumn_2025 LIMIT 1000

In [0]:
-- schools autumn 2025
SELECT * FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2025 LIMIT 1000

### Data Manipulation
Select required columns and metadata, join pupils to schools.

In [0]:
-- Pupils: select columns 
CREATE OR REPLACE TEMP VIEW pupils_aut AS
SELECT 
  pupil_id,
  gender,
  school_urn,
  metadata_json:address:postcode AS postcode,
  LOWER(metadata_json:sen_status) AS sen_status,
  metadata_json:fsm_eligible AS fsm_status
FROM catalog_40_copper_analyst_training.bronze.pupils_autumn_2025;

-- Schools: select columns, title-case city, deduplicate
CREATE OR REPLACE TEMP VIEW schools_aut AS
SELECT DISTINCT
  school_urn,
  INITCAP(city) AS city,
  school_type,
  REPLACE(metadata_json:ofsted_rating, '"', '') AS ofsted_rating,
  CAST(metadata_json:pupil_premium_pct AS DOUBLE) AS pupil_premium_pct
FROM catalog_40_copper_analyst_training.bronze.schools_autumn_2025;

-- Join pupils and schools
CREATE OR REPLACE TEMP VIEW pupils_schools_aut AS
SELECT 
  p.*,
  s.city,
  s.school_type,
  s.ofsted_rating,
  s.pupil_premium_pct
FROM pupils_aut p
LEFT JOIN schools_aut s
  ON p.school_urn = s.school_urn

In [0]:
-- Investigate relationships in joined data
SELECT gender, ofsted_rating, COUNT(*) AS n
FROM pupils_schools_aut
GROUP BY gender, ofsted_rating
ORDER BY n DESC

### Outputs
Produce the summary output:
- Average pupil premium percentage by Ofsted rating

In [0]:
-- Output: avg_pupil_premium_by_ofsted
SELECT 
  ofsted_rating,
  COUNT(*) AS pupils,
  ROUND(AVG(pupil_premium_pct), 4) AS avg_pupil_premium_pct,
  ROUND(PERCENTILE(pupil_premium_pct, 0.5), 4) AS median_pupil_premium_pct
FROM pupils_schools_aut
GROUP BY ofsted_rating
ORDER BY pupils DESC